In [ ]:
# ============================================================
# RELIABLE GOOGLE DRIVE MOUNT
# ============================================================

import os
import shutil
from google.colab import drive

MOUNT_POINT = "/content/drive"

# Case 1: Drive is already properly mounted
if os.path.ismount(MOUNT_POINT):
    print("Google Drive is already mounted.")

# Case 2: The folder exists but is only a stale, unmounted folder
else:
    if os.path.exists(MOUNT_POINT):
        print("Removing stale /content/drive folder...")
        raise RuntimeError("Choose a fresh Colab runtime before mounting Drive.")

    # Create a clean mount point
    os.makedirs(MOUNT_POINT, exist_ok=True)

    # Mount Google Drive
    drive.mount(MOUNT_POINT)

# Final verification
if os.path.exists("/content/drive/MyDrive"):
    print("Google Drive is ready.")
    print("Mounted status:", os.path.ismount(MOUNT_POINT))
else:
    raise RuntimeError("Google Drive mounting failed.")

In [ ]:
import pandas as pd

# Load final cognitive dataset
final_cognitive = pd.read_csv(
    "/content/drive/MyDrive/ADNI_Project/Cognitive_Data/final_cognitive_702.csv"
)

print(final_cognitive.shape)
final_cognitive.head()

In [ ]:
final_cognitive.isnull().sum()

In [ ]:
print(final_cognitive.columns.tolist())

In [ ]:
# Create feature matrix (X)
X = final_cognitive.drop(columns=["RID", "VISCODE", "DIAGNOSIS"])

# Create target variable (y)
y = final_cognitive["DIAGNOSIS"]

print("Feature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)

print("\nFeature Columns:")
print(X.columns.tolist())

In [ ]:
print(y.unique())
print(y.value_counts().sort_index())

In [ ]:
# Convert diagnosis labels from float to integer
y = y.astype(int)

print("Target Data Type:", y.dtype)
print("Unique Labels:", sorted(y.unique()))

In [ ]:
from sklearn.model_selection import train_test_split

# Split the data into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)

print("Training Labels:", y_train.shape)
print("Testing Labels:", y_test.shape)

In [ ]:
print("Original Dataset")
print(y.value_counts(normalize=True).sort_index())

print("\nTraining Set")
print(y_train.value_counts(normalize=True).sort_index())

print("\nTesting Set")
print(y_test.value_counts(normalize=True).sort_index())

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Create Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train the model
rf_model.fit(X_train, y_train)

print("Random Forest model trained successfully!")

In [ ]:
# Predict the diagnosis for the test set
y_pred = rf_model.predict(X_test)

print("Predictions completed!")
print(y_pred[:10])

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")

print("\nClassification Report")
print(classification_report(
    y_test,
    y_pred,
    target_names=["CN", "MCI", "AD"]
))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred))

In [ ]:
# ============================================================
# RANDOM FOREST CONFUSION MATRIX
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Generate predictions
y_pred = rf_model.predict(X_test)

# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)

class_names = [
    "CN",
    "MCI",
    "AD"
]

fig, ax = plt.subplots(figsize=(6.2, 5.5))

image = ax.imshow(
    cm,
    interpolation="nearest",
    cmap="Blues"
)

# Colour bar
colour_bar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
colour_bar.set_label("Number of Participants", fontsize=11)

# Axis labels and ticks
ax.set(
    xticks=np.arange(len(class_names)),
    yticks=np.arange(len(class_names)),
    xticklabels=class_names,
    yticklabels=class_names,
    xlabel="Predicted Diagnosis",
    ylabel="Actual Diagnosis",
    title="Random Forest Confusion Matrix"
)

ax.set_title(
    "Random Forest Confusion Matrix",
    fontsize=17,
    fontweight="bold",
    pad=18
)

ax.set_xlabel(
    "Predicted Diagnosis",
    fontsize=12,
    fontweight="bold",
    labelpad=12
)

ax.set_ylabel(
    "Actual Diagnosis",
    fontsize=12,
    fontweight="bold",
    labelpad=12
)

ax.tick_params(axis="both", labelsize=11)

# Add values inside cells
threshold = cm.max() / 2

for row in range(cm.shape[0]):
    for column in range(cm.shape[1]):

        value = cm[row, column]

        ax.text(
            column,
            row,
            f"{value}",
            ha="center",
            va="center",
            fontsize=16,
            fontweight="bold",
            color="white" if value > threshold else "#1f2937"
        )

# Add subtle grid lines between cells
ax.set_xticks(np.arange(-0.5, len(class_names), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(class_names), 1), minor=True)

ax.grid(
    which="minor",
    linewidth=1.5
)

ax.tick_params(which="minor", bottom=False, left=False)

# Add participant summary below the graph
class_totals = cm.sum(axis=1)
correct_counts = np.diag(cm)
class_recalls = correct_counts / class_totals

summary_text = (
    f"CN correctly classified: {correct_counts[0]}/{class_totals[0]} "
    f"({class_recalls[0] * 100:.1f}%)\n"
    f"MCI correctly classified: {correct_counts[1]}/{class_totals[1]} "
    f"({class_recalls[1] * 100:.1f}%)\n"
    f"AD correctly classified: {correct_counts[2]}/{class_totals[2]} "
    f"({class_recalls[2] * 100:.1f}%)"
)

fig.text(
    0.5,
    0.01,
    summary_text,
    ha="center",
    va="bottom",
    fontsize=10.5,
    bbox={
        "boxstyle": "round,pad=0.6",
        "facecolor": "#f3f8fc",
        "edgecolor": "#9ec5e5"
    }
)

plt.tight_layout(rect=[0, 0.11, 1, 1])

# Save a high-resolution dissertation image
plt.savefig(
    "random_forest_confusion_matrix_professional.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
#  RANDOM FOREST FEATURE IMPORTANCE GRAPH
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# Use X_train column names if available
feature_names = list(X_train.columns)

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=True
)

fig, ax = plt.subplots(figsize=(7, 6))

# Create a refreshing gradient palette
normalised_values = np.linspace(0.15, 0.95, len(importance_df))
bar_colours = cm.viridis_r(normalised_values)

bars = ax.barh(
    importance_df["Feature"],
    importance_df["Importance"],
    color=bar_colours,
    edgecolor="white",
    linewidth=0.8,
    height=0.72
)

ax.set_title(
    "Random Forest Feature Importance",
    fontsize=17,
    fontweight="bold",
    pad=20
)

ax.set_xlabel(
    "Relative Contribution",
    fontsize=12,
    fontweight="bold",
    labelpad=10
)

ax.set_ylabel(
    "Cognitive and Demographic Feature",
    fontsize=12,
    fontweight="bold",
    labelpad=10
)

ax.tick_params(axis="both", labelsize=10)

# Add light vertical grid lines
ax.xaxis.grid(
    True,
    linestyle="--",
    linewidth=0.8,
    alpha=0.35
)

ax.set_axisbelow(True)

# Remove unnecessary borders
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

# Add importance scores at the end of each bar
maximum_importance = importance_df["Importance"].max()

for bar, value in zip(bars, importance_df["Importance"]):

    ax.text(
        value + maximum_importance * 0.015,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.4f}",
        va="center",
        fontsize=9.5,
        fontweight="bold",
        color="#1f2937"
    )

ax.set_xlim(
    0,
    maximum_importance * 1.18
)

# Identify the three most influential features
top_three = (
    importance_df
    .sort_values("Importance", ascending=False)
    .head(3)["Feature"]
    .tolist()
)

insight_text = (
    "Most influential features: "
    + ", ".join(top_three)
)

fig.text(
    0.5,
    0.015,
    insight_text,
    ha="center",
    fontsize=10.5,
    fontweight="bold",
    bbox={
        "boxstyle": "round,pad=0.6",
        "facecolor": "#f1fbf6",
        "edgecolor": "#9bd3b1"
    }
)

plt.tight_layout(rect=[0, 0.07, 1, 1])

# Save a high-resolution dissertation image
plt.savefig(
    "random_forest_feature_importance_professional.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
!pip install xgboost

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
y_train_xgb = y_train - 1
y_test_xgb = y_test - 1

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    eval_metric="mlogloss"
)

xgb_model.fit(X_train, y_train_xgb)

print("XGBoost model trained successfully!")

In [ ]:
y_pred_xgb = xgb_model.predict(X_test)

print(y_pred_xgb[:10])

In [ ]:
accuracy_xgb = accuracy_score(y_test_xgb, y_pred_xgb)

print(f"XGBoost Accuracy: {accuracy_xgb:.4f}")

print("\nClassification Report")
print(classification_report(
    y_test_xgb,
    y_pred_xgb,
    target_names=["CN", "MCI", "AD"]
))

print("\nConfusion Matrix")
print(confusion_matrix(y_test_xgb, y_pred_xgb))

In [ ]:
# ============================================================
# SAVE THE COGNITIVE RANDOM FOREST MODEL
# ============================================================

import os
import joblib

# Folder in Google Drive where interface model files will be stored
MODEL_FOLDER = "/content/drive/MyDrive/ADNI_Project/Interface_Models"

# Create the folder if it does not already exist
os.makedirs(MODEL_FOLDER, exist_ok=True)

# Preserve the exact feature order used during model training
cognitive_feature_columns = [
    "MMSCORE",
    "DELAYED_RECALL",
    "CDMEMORY",
    "CDORIENT",
    "CDJUDGE",
    "CDCOMMUN",
    "CDHOME",
    "CDCARE",
    "CDGLOBAL",
    "CDRSB",
    "TOTSCORE",
    "TOTAL13",
    "FAQTOTAL",
    "GENDER",
    "PTEDUCAT",
    "AGE"
]

# Save the trained Random Forest cognitive model
cognitive_model_path = os.path.join(
    MODEL_FOLDER,
    "cognitive_random_forest.joblib"
)

joblib.dump(rf_model, cognitive_model_path)

# Save the feature order separately
cognitive_features_path = os.path.join(
    MODEL_FOLDER,
    "cognitive_feature_columns.joblib"
)

joblib.dump(cognitive_feature_columns, cognitive_features_path)

print("Cognitive model saved successfully.")
print("Model path:", cognitive_model_path)
print("Feature columns path:", cognitive_features_path)

In [ ]:
# ============================================================
# VERIFY SAVED COGNITIVE MODEL FILES
# ============================================================

import os

files = os.listdir(
    "/content/drive/MyDrive/ADNI_Project/Interface_Models"
)

print("Files in Interface_Models folder:")

for file in files:
    print("-", file)